In [1]:
from gen_cluster_mitigationplan import *

# ==================== TEST CODE =====================
dir_path = os.getcwd()
pcg_data_path = f"{dir_path}/nodes_and_edges_PCG.pkl"
lotus_south_data_path = f"{dir_path}/nodes_and_edges_lotus_south.pkl"
# data_path_dict = {"PCG": pcg_data_path, "Lotus South": lotus_south_data_path}
data_path_dict = {"PCG": pcg_data_path}

all_data_df, clusters, G, nodes = find_graph_properties(data_path_dict)

risk_high = all_data_df[
    all_data_df["risk_level"] >= 3
]  # TODO: need to handle the edge case when there's no high/critical risk level
# TopN risks with a given property (central/source/sink)
N_TOP = 3
risk_central = top_n_with_row_limit(
    all_data_df, "betweenness_centrality_non_weight", n=N_TOP
)
risk_source = top_n_with_row_limit(all_data_df, "out_degree", n=N_TOP)

# ======= cluster by type (high/central/source risk) =======

# cluster(s) that contains risk_high
highrisk_clusters = find_sublists_with_any(risk_high["risk_id"].tolist(), clusters)
# print(highrisk_clusters)

# cluster(s) that contains risk_central
centralrisk_clusters = find_sublists_with_any(
    risk_central["risk_id"].tolist(), clusters
)
# print(centralrisk_clusters)

# cluster(s) that contains risk_source
sourcerisk_clusters = find_sublists_with_any(risk_source["risk_id"].tolist(), clusters)
# print(sourcerisk_clusters)

# ======= ALL important clusters (contains high/central/source risks) =======
combined_list = list(
    set(
        risk_high["risk_id"].tolist()
        + risk_central["risk_id"].tolist()
        + risk_source["risk_id"].tolist()
    )
)
important_clusters = find_sublists_with_any(combined_list, clusters)

# # === STEP 5: OUTPUT PROMPTS ===
# for i, cluster in enumerate(important_clusters, 1):
#     print(f"\n--- Cluster {i} Prompt ---\n")
#     print(generate_prompt_nointro(cluster, G, nodes))

# print(generate_prompt_nointro(important_clusters[0], G, nodes))

In [5]:
# from dotenv import load_dotenv
# import os
# from typing import List, Tuple
from langchain_community.callbacks import get_openai_callback

# TODO: need to acquire the api_key from your directory
# load_dotenv("../../.env")

# estimate_cost.total_cost_THB = 0  # initialization
usage_count_list = []

# notice, this is a single cluster
cluster_risk_to_plan = generate_prompt_nointro(important_clusters[1], G, nodes)
# cluster_risk_to_plan = generate_prompt_nointro(highrisk_clusters, G, nodes)

# gen 'cluster' mitigation plans
# response = get_response_control_cluster(cluster_risk_to_plan)  # accept single cluster
with get_openai_callback() as cb:
    response = get_response_control_cluster(
        cluster_risk_to_plan
    )  # accept single cluster
    print(cb)
    thb = cb.total_cost * 35
    print(f"total cost (THB): {thb}")

# response = get_response_test(cluster_risk_to_plan)

# usage_count = response.usage
# usage_count_list.append(usage_count)
# MODEL = "gpt-4.1"  # "gpt-4o"
# estimate_cost(res_usage=usage_count_list, type="multi", model=MODEL)

Tokens Used: 1810
	Prompt Tokens: 1211
		Prompt Tokens Cached: 1024
	Completion Tokens: 599
		Reasoning Tokens: 0
Successful Requests: 1
Total Cost (USD): $0.0077375
total cost (THB): 0.2708125


In [ ]:
# response.choices[0].message.content

"Hello! I'm just a virtual assistant, so I don't have feelings, but I'm here and ready to help you. How can I assist you today?"

In [ ]:
# save to json
tmp_res = response.choices[0].message.content
report_json = json.loads(tmp_res)

data = []
data.append(report_json)

# input_file = report_json_folder + str(selected_year) + '_Q' + str(selected_quarter) + '_' + selected_company_report + '_json_riskcontrol.json'
file_path = f"{dir_path}/clustercontrol.json"

with open(file_path, "w") as file:
    json.dump(data, file, indent=4, ensure_ascii=False)

report_json